# ccphit scratch

Ad-hoc probes and REPL-style experiments. Run from repo root:

```bash
cd ~/Documents/ccphit
uv run jupyter lab notebooks/scratch.ipynb
```

In [1]:
from pathlib import Path

# Jupyter's cwd is usually the notebook dir; project code expects repo root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    import os
    os.chdir(ROOT)

ROOT

PosixPath('/Users/jesus/Documents/ccphit')

In [ ]:
import geopandas as gpd
import pandas as pd

from ccphit.config import load_config
from ccphit.sources.cooling_centers import fetch_cooling_centers

In [3]:
config = load_config()
config["sources"]["cooling"]["url"]

'https://services.arcgis.com/RmCCgQtiZLDCtblq/ArcGIS/rest/services/Cooling_Centers_-_Los_Angeles_County_Regional_(Official_-_Updated_July_2022)_(View)/FeatureServer/0/query'

In [4]:
cooling = fetch_cooling_centers(config)
cooling.info()
cooling.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 168 entries, 166 to 117
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   site_name                168 non-null    str     
 1   address                  168 non-null    str     
 2   days_hours_of_operation  168 non-null    str     
 3   geometry                 168 non-null    geometry
dtypes: geometry(1), str(3)
memory usage: 31.6 KB


,site_name,address,days_hours_of_operation,geometry
166,Walteria Library,3815 242nd Street,Monday - Friday 10 a.m. – 6 p.m.\nSaturday 9am...,POINT (-118.35303 33.80454)
98,Lomita Library,24200 Narbonne Avenue,Tuesday-Wednesday: 12 pm - 8 pm\nThursday-Satu...,POINT (-118.31913 33.8054)
151,Southeast Library,23115 Arlington Ave,Monday - Friday 10 a.m. – 6 p.m. ...,POINT (-118.32019 33.81536)
56,El Retiro Library,126 Vista Del Parque,"Monday, Wednesday & Friday: 10:00 a.m. - 2:00 ...",POINT (-118.37888 33.81564)
40,Dee Hardison Sports Center,2400 Jefferson Street,Monday - Friday 9 a.m. – 10 p.m.\nSaturday 8 a...,POINT (-118.32434 33.82689)


---
Scratch space below — API probes, quick plots, column tidying, etc.

In [5]:
cooling = gpd.read_parquet("data/processed/cooling_centers.parquet")
heat = pd.read_parquet("data/processed/heat_scores.parquet")
print(cooling.crs, len(cooling))
print(heat.columns.tolist(), len(heat))
print(heat["heat_risk"].value_counts())


{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal

In [9]:
m = cooling.explore(tooltip=["site_name", "address"], color="#e4572e")
m.save("cooling.html")

In [15]:
la_zcta = gpd.read_parquet("data/processed/zcta_bounds.parquet")
m = la_zcta.explore(tooltip=["zcta"], color="#e4572e")
m.save("zcta.html")


In [16]:
la_zcta.head()

,geometry,zcta,GEOID,POP100
1,"POLYGON ((-118.53739 34.13313, -118.53451 34.1...",91316,91316,28777
6,"POLYGON ((-118.51417 34.32949, -118.50876 34.3...",91342,91342,96429
8,"POLYGON ((-118.47218 34.25101, -118.47216 34.2...",91345,91345,18895
9,"POLYGON ((-118.38278 34.02043, -118.38205 34.0...",90016,90016,46512
10,"POLYGON ((-118.30084 34.03218, -118.3006 34.03...",90007,90007,40944


In [ ]:
zs = gpd.read_parquet("data/processed/zcta_svi.parquet").explore(
    column="svi", cmap="Reds", scheme="quantiles", k=5, legend=True)

zs.save("zcta_svi.html")
